# Proposed tools


### Initializing candidate in EXPLOG
...

------

### Submit process:

Args:

- `candidate_id`
- `calculation_type`
- `termination_index = None`
- `site_index = None`

------

### Update log (process log):

- For updating the log - and importantly getting info on any changes in the log

(No Args)

------

### List initial sites for termination:

Gives df of adsorbtion sites - before the surface is relaxed and thus these sites are pleliminary

Args:
- `candidate_id`
- `termination_index`

Optional Args:
- `only_reduced_coord_O_sites = True`

------

### List (actual) sites for termination:

Gives df of adsorbtion sites - after the surface is relaxed 

Args:
- `candidate_id`
- `termination_index`

Optional Args:
- `only_reduced_coord_O_sites = True`

------



Other tools will be needed to display contences of log etc....



In [44]:
# --- standard library imports ---
import numpy as np
import pandas as pd
from pathlib import Path

from ase import Atoms
from ase.io import read, Trajectory
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor

# --- IMPORTS FROM THIS PROJECT ---
from gnome_dreams_oer_screening import OER_catalyst_study
from gnome_dreams_oer_screening import Explog

In [45]:
EXPLOG = Explog()
EXPLOG.init('TEMP_vasp_calcs', 'test')

In [46]:
# Test structures, to hand to the study object:
ZrO2_m = read('ZrO2_m.cif')
TiO2_r = read('TiO2_rutile.cif')
test_cands = Trajectory('test_cands.traj')

atoms = test_cands[18]
S = AseAtomsAdaptor.get_structure(atoms)

In [ ]:
# --- TOOL: Initializing candidate in the experiment log ---------------
# ______________________________________________________________________

study = OER_catalyst_study(atoms,
                            H2O_gas_free_energy = -14.217,
                            H2_gas_free_energy = -6.77
                           )
cant_id = "07286qnfd"

EXPLOG.add_candidate(
    candidate_id=cant_id,
    reason_or_hypothesis="Ir sites should be great", 
    notes="this is a great candidate", 
    study_obj=study
)
# ----------------------------------------------------------------------

EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,07286qnfd,Ir sites should be great,this is a great candidate,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,<NA>


In [48]:
# --- TOOL: Entering a process in the explog and submitting it ---------
# ______________________________________________________________________

# Args:
candidate_id = cant_id
calculation_type = "bulk_relaxation"
termination_index = None
site_index = None

"""
NOTE:
Logic needed for initializing surface_adsorbtion_studies and
adsorbtion_site_studies HERE! - see below for details
"""

p_id = EXPLOG.add_process(
    candidate_id=cant_id, 
    calculation_type="bulk_relaxation",
    termination_index=termination_index,
    site_index=site_index
    )

# TEMP LINE:
print(EXPLOG.relational_frame.processes.df, '\n', '-'*60)

EXPLOG.submit_process(process_id=p_id)

# TEMP LINE:
print(EXPLOG.relational_frame.processes.df, '\n', '-'*60)

# ----------------------------------------------------------------------

  process_id candidate_id         job_type slurmID        status  \
0          0    07286qnfd  bulk_relaxation    <NA>  un-submitted   

  termination_index site_index  \
0              None       None   

                                            VASP_dir processNote  
0  TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...        None   
 ------------------------------------------------------------
  process_id candidate_id         job_type slurmID   status termination_index  \
0          0    07286qnfd  bulk_relaxation      42  pending              None   

  site_index                                           VASP_dir processNote  
0       None  TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...        None   
 ------------------------------------------------------------


In [ ]:
# --- TOOL: Updating the process log (EXPLOG) --------------------------
# ______________________________________________________________________

updates_dict = EXPLOG.update_log()

updates_dict # dict with process_id as key and new status as value

# ----------------------------------------------------------------------

{'0': 'completed'}

In [50]:
# Just to show the process log has been updated after the update_log() 
# call:
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,42,completed,None,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...,None


In [ ]:
# --- TOOL: ranking surfaces, and presenting the ranking df ------------
# ______________________________________________________________________

"""
NOTE:
This tool should ideally have an timeout, as some struces will not yeild
surfaces - and atempting to generate them will take a long time.
"""

# Args:
cant_id = cant_id
max_miller = 1      # Max miller index

# Args to leve out (for now):
method = 'all'      # What coordination to consider
stoichiometry_tolerance = 0.2 # Allowed stoichiometry diviation from bulk
all_species_present = True # Only surfaces with all bulk species present
symmetrize = True # Whether to symmetrize the surfaces
select_closest_O_stoichiometry = True # Select surfaces with closest O stoichiometry to bulk
min_slab_thickness = 9 # Minimum slab thickness in Å
max_slab_thickness = 20 # Maximum slab thickness in Å
min_atoms = 20 # Minimum number of atoms in the slab
max_atoms = 120 # Maximum number of atoms in the slab
max_layers = 6 # Maximum number of layers, considered when building the slabs

study = EXPLOG.relational_frame.candidates[cant_id].study_obj
ranking = study.get_termination_rankings() # None on first call
if ranking is None:
    # No arguments needs to be provided (all have defaults as shown above):
    study.predict_most_likely_surfaces(
    max_miller = max_miller,
    method = method,
    stoichiometry_tolerance = stoichiometry_tolerance,
    all_species_present = all_species_present,
    symmetrize = symmetrize,
    select_closest_O_stoichiometry = select_closest_O_stoichiometry,
    min_slab_thickness = min_slab_thickness,
    max_slab_thickness = max_slab_thickness,
    min_atoms = min_atoms,
    max_atoms = max_atoms,
    max_layers = max_layers
    )
    ranking = study.get_termination_rankings()

ranking.sort_values('Normalized score', ascending=False)

# ----------------------------------------------------------------------

,Miller indices,Termination index,Score,Normalized score
10,"(0, 1, 0)",10,-0.064210,1.000000
0,"(1, 1, 1)",0,-0.066443,0.943006
1,"(1, 1, 0)",1,-0.076023,0.698460
9,"(1, -1, 0)",9,-0.079629,0.606405
8,"(1, -1, 0)",8,-0.079629,0.606405
5,"(1, 0, -1)",5,-0.081684,0.553936
4,"(1, 0, -1)",4,-0.081684,0.553936
7,"(1, -1, 1)",7,-0.101535,0.047209
6,"(1, -1, 1)",6,-0.101535,0.047209
2,"(1, 1, -1)",2,-0.103384,0.000000


In [ ]:
# --- TOOL: List sites for termination (before its initilization) ------
# ______________________________________________________________________

# Args:
cand_id = cant_id # will be needed to get the study object...
termination_index = 0 # What termination to yield the adsorption sites for

# Arg which may be left out (for now):
only_reduced_coord_O_sites = True # Use True as default for now

df = study.get_init_adsorption_sites_df(termination_index, 
                only_reduced_coord_O_sites=only_reduced_coord_O_sites)

# ----------------------------------------------------------------------

# Just for viewing the structure ourselves:
if False:
    view(study.get_init_adsorption_sites_atoms(termination_index,
            only_reduced_coord_O_sites=only_reduced_coord_O_sites))

df # just to show - what needs to be shown to the agent

,site type,ad site element,ad site neighboring elements,reduced coordination,position,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH)
0,on-top,Zn,"[(O, 1.9660994528238742), (O, 1.96609945282387...",N/A,"[2.2085651522850935, 2.9549730780970673, 21.36...",None,None,None,None,None,None
1,on-top,Bi,"[(O, 2.3097913267635857), (O, 2.30979132676358...",N/A,"[-1.3768128757148554, 2.974504547726728, 21.17...",None,None,None,None,None,None
2,on-top,Bi,"[(O, 2.309791122057748), (O, 2.309791122057748...",N/A,"[0.5197245157172787, 6.117745845952744, 21.179...",None,None,None,None,None,None
3,on-top,Ir,"[(O, 1.961347524109922), (O, 1.961347524109930...",N/A,"[-3.024567666731645, 6.112487373360143, 20.663...",None,None,None,None,None,None
4,lattice O,N/A,"[(Ir, 1.9613475241099303), (Bi, 2.502006429151...",1,"[-1.456098585485744, 5.16612059850852, 19.8415...",None,None,None,None,None,None
5,lattice O,N/A,"[(Ir, 1.991253888095814), (Bi, 2.4599197427068...",1,"[2.718972543684037, 5.190163251198649, 20.3768...",None,None,None,None,None,None
6,lattice O,N/A,"[(Ir, 1.9912537872783354), (Bi, 2.459919651991...",1,"[0.4691178223776624, 1.4613486720783215, 20.37...",None,None,None,None,None,None


In [53]:
study.get_surface_studies()

{}

In [54]:
termination_index = 0 # What termination to initialize and submit

# --- Logic for termination initialization: ----------------------------
surface_studies = study.get_surface_studies()
if termination_index not in surface_studies.keys():
    study.initialize_oer_surface_study(termination_index)
# ----------------------------------------------------------------------

p_id = EXPLOG.add_process( 
    candidate_id = cant_id,
    calculation_type = "surface_relaxation",
    termination_index = termination_index
)

# TEMP LINE:
print(EXPLOG.relational_frame.processes.df, '\n', '-'*60)

EXPLOG.relational_frame.processes.df

EXPLOG.submit_process(process_id = p_id)

# TEMP LINE:
print(EXPLOG.relational_frame.processes.df, '\n', '-'*60)
# ----------------------------------------------------------------------

  process_id candidate_id            job_type slurmID        status  \
0          0    07286qnfd     bulk_relaxation      42     completed   
1          1    07286qnfd  surface_relaxation    <NA>  un-submitted   

  termination_index site_index  \
0              None       None   
1                 0       None   

                                            VASP_dir processNote  
0  TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...        None  
1  TEMP_vasp_calcs\20260217_171309_07286qnfd_surf...        None   
 ------------------------------------------------------------
  process_id candidate_id            job_type slurmID     status  \
0          0    07286qnfd     bulk_relaxation      42  completed   
1          1    07286qnfd  surface_relaxation      43    pending   

  termination_index site_index  \
0              None       None   
1                 0       None   

                                            VASP_dir processNote  
0  TEMP_vasp_calcs\20260217_171309_07286qnfd

In [55]:
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,42,completed,None,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,43,pending,0,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_surf...,None


In [56]:
# Tool for this was considered above (but update call needed inbetween):
EXPLOG.update_log()
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,42,completed,None,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,43,completed,0,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_surf...,None


In [57]:
# --- TOOL: Viewing sites for termination - after relaxation i.e., the 
# actual sites ---------------------------------------------------------
# ______________________________________________________________________

# Args:
cand_id = cant_id # will be needed to get the study object...
termination_index = 0 # What termination to yield the adsorption sites for
only_reduced_coord_O_sites = True # Use True as default for now

study = EXPLOG.relational_frame.candidates.df['study_obj'][0]
surface_study = study.get_surface_studies()[0]

# NOTE: can only determine adsorbtions sites onece!
if surface_study.get_adsorption_sites_df() is None:
    surface_study.determine_adsorption_sites(
        only_reduced_coord_O_sites = only_reduced_coord_O_sites)
    
surface_study.get_adsorption_sites_df() # DF to display to agent
# ----------------------------------------------------------------------

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH)
0,0,on-top,Zn,"[(O, 1.9660994528238716), (O, 1.96609945282388...",N/A,"[2.208565152285095, 2.9549730780970673, 27.168...",None,None,None,None,None,None
1,1,on-top,Bi,"[(O, 2.3097913267635857), (O, 2.30979132676358...",N/A,"[-1.3768128757148554, 2.974504547726728, 26.97...",None,None,None,None,None,None
2,2,on-top,Bi,"[(O, 2.3097911220577467), (O, 2.30979112205774...",N/A,"[0.519724515717281, 6.117745845952744, 26.9783...",None,None,None,None,None,None
3,3,on-top,Ir,"[(O, 1.9613475241099239), (O, 1.96134752410992...",N/A,"[-3.024567666731644, 6.112487373360143, 26.461...",None,None,None,None,None,None
4,4,lattice O,N/A,"[(Ir, 1.961347524109927), (Bi, 2.5020064291512...",1,"[-1.4560985854857442, 5.16612059850852, 25.640...",None,None,None,None,None,None
5,5,lattice O,N/A,"[(Ir, 1.9912538880958124), (Bi, 2.459919742706...",1,"[2.718972543684037, 5.190163251198649, 26.1753...",None,None,None,None,None,None
6,6,lattice O,N/A,"[(Ir, 1.9912537872783338), (Bi, 2.459919651991...",1,"[0.4691178223776631, 1.4613486720783215, 26.17...",None,None,None,None,None,None


In [58]:
ad_site_index = 0 # What adsorption site to initialize and submit

# --- Logic for adsorbtion site study initialization: ------------------
study = EXPLOG.relational_frame.candidates.df['study_obj'][0]
surface_study = study.get_surface_studies()[0]
ad_site_studies_dict = surface_study.get_adsorption_site_studies_dict()
if ad_site_index not in ad_site_studies_dict.keys():
    surface_study.initialize_adsorption_site_study(ad_site_index)
# ----------------------------------------------------------------------

p_id = EXPLOG.add_process( 
    candidate_id = cant_id,
    calculation_type = "O_adsorption",
    termination_index = termination_index,
    site_index = ad_site_index
)

EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,42,completed,None,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,43,completed,0,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_surf...,None
2,2,07286qnfd,O_adsorption,<NA>,un-submitted,0,0,TEMP_vasp_calcs\20260217_171309_07286qnfd_O_ad...,None


In [59]:
EXPLOG.submit_process(process_id = p_id)
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,42,completed,None,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,43,completed,0,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_surf...,None
2,2,07286qnfd,O_adsorption,44,pending,0,0,TEMP_vasp_calcs\20260217_171309_07286qnfd_O_ad...,None


In [60]:
EXPLOG.update_log()
EXPLOG.relational_frame.processes.df

,process_id,candidate_id,job_type,slurmID,status,termination_index,site_index,VASP_dir,processNote
0,0,07286qnfd,bulk_relaxation,42,completed,None,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_bulk...,None
1,1,07286qnfd,surface_relaxation,43,completed,0,None,TEMP_vasp_calcs\20260217_171309_07286qnfd_surf...,None
2,2,07286qnfd,O_adsorption,44,completed,0,0,TEMP_vasp_calcs\20260217_171309_07286qnfd_O_ad...,None


In [61]:
EXPLOG.relational_frame.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,07286qnfd,Ir sites should be great,this is a great candidate,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,1.25


In [62]:
if False:
    view(surface_study.get_relaxed_surface())

<Popen: returncode: None args: ['c:\\Users\\matnis\\OneDrive - Danmarks Tekn...>

In [63]:
surface_study.update_adsorption_energies()
surface_study.get_adsorption_sites_df()

,Site index,site type,ad site element,ad site neighboring elements,reduced coordination,position,G_approx(O),G(O),G_approx(OH),G(OH),G_approx(OOH),G(OOH)
0,0,on-top,Zn,"[(O, 1.9660994528238716), (O, 1.96609945282388...",N/A,"[2.208565152285095, 2.9549730780970673, 27.168...",3.712868,None,None,None,None,None
1,1,on-top,Bi,"[(O, 2.3097913267635857), (O, 2.30979132676358...",N/A,"[-1.3768128757148554, 2.974504547726728, 26.97...",None,None,None,None,None,None
2,2,on-top,Bi,"[(O, 2.3097911220577467), (O, 2.30979112205774...",N/A,"[0.519724515717281, 6.117745845952744, 26.9783...",None,None,None,None,None,None
3,3,on-top,Ir,"[(O, 1.9613475241099239), (O, 1.96134752410992...",N/A,"[-3.024567666731644, 6.112487373360143, 26.461...",None,None,None,None,None,None
4,4,lattice O,N/A,"[(Ir, 1.961347524109927), (Bi, 2.5020064291512...",1,"[-1.4560985854857442, 5.16612059850852, 25.640...",None,None,None,None,None,None
5,5,lattice O,N/A,"[(Ir, 1.9912538880958124), (Bi, 2.459919742706...",1,"[2.718972543684037, 5.190163251198649, 26.1753...",None,None,None,None,None,None
6,6,lattice O,N/A,"[(Ir, 1.9912537872783338), (Bi, 2.459919651991...",1,"[0.4691178223776631, 1.4613486720783215, 26.17...",None,None,None,None,None,None
